# Gold Price Prediction using LSTM-Attention
**Undergraduate ML Project** — Run on Google Colab (T4 GPU)

## Pipeline Overview

```
Data Collection → Feature Engineering → Scaling →
Walk-Forward CV (5 folds) → Best Model Selection →
Bias Correction → Evaluation → Visualizations
```

**Local deliverables** (not in Colab):
- Flask web dashboard: `py app.py` → http://127.0.0.1:5000
- CLI inference: `py -m src.predict_future [--days N]`

Just click **Runtime → Run all** and it runs end-to-end.

In [ ]:
# ===== CELL 1: Install Dependencies =====
!pip install yfinance pandas numpy scikit-learn torch matplotlib seaborn flask joblib -q

In [ ]:
# ===== CELL 2: Imports + Device Setup =====
import os, warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ===== CELL 3: Data Collection =====
# Fetches GLD (Gold ETF) data from Yahoo Finance

print("=" * 60)
print("GOLD PRICE PREDICTION USING LSTM-ATTENTION (PyTorch)")
print("=" * 60)

ticker = yf.Ticker("GLD")
df = ticker.history(start="2015-01-01", interval="1d")

# Flatten multi-level column names
df.columns = [c.lower() if isinstance(c, str) else '_'.join(c).lower() for c in df.columns]

# Keep only OHLCV columns
df = df[["open", "high", "low", "close", "volume"]]

# Clean: forward-fill, back-fill, sort, remove duplicates
df = df.ffill().bfill().sort_index()
df = df[~df.index.duplicated(keep='first')]

print(f"Retrieved {len(df)} records")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

In [ ]:
# ===== CELL 4: Feature Engineering =====
# Creates 31 technical indicator features from raw OHLCV data

d = df.copy()

# -- Moving Averages --
d['sma_10'] = d['close'].rolling(10).mean()
d['sma_20'] = d['close'].rolling(20).mean()
d['sma_50'] = d['close'].rolling(50).mean()
d['ema_10'] = d['close'].ewm(span=10, adjust=False).mean()
d['ema_20'] = d['close'].ewm(span=20, adjust=False).mean()

# -- Moving Average Crossovers --
d['sma_cross_10_20'] = d['sma_10'] - d['sma_20']
d['sma_cross_20_50'] = d['sma_20'] - d['sma_50']

# -- Price Returns (our target variable) --
d['returns'] = d['close'].pct_change()  # Target: daily % change
d['log_returns'] = np.log(d['close'] / d['close'].shift(1))

# -- Volatility --
d['volatility_10'] = d['returns'].rolling(10).std()
d['volatility_20'] = d['returns'].rolling(20).std()

# -- RSI (Relative Strength Index) --
delta = d['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss
d['rsi_14'] = 100 - (100 / (1 + rs))

# -- MACD (Moving Average Convergence Divergence) --
ema_12 = d['close'].ewm(span=12, adjust=False).mean()
ema_26 = d['close'].ewm(span=26, adjust=False).mean()
d['macd'] = ema_12 - ema_26
d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
d['macd_hist'] = d['macd'] - d['macd_signal']

# -- Bollinger Bands --
d['bb_middle'] = d['close'].rolling(20).mean()
bb_std = d['close'].rolling(20).std()
d['bb_upper'] = d['bb_middle'] + (bb_std * 2)
d['bb_lower'] = d['bb_middle'] - (bb_std * 2)
d['bb_width'] = (d['bb_upper'] - d['bb_lower']) / d['bb_middle']
d['bb_position'] = (d['close'] - d['bb_lower']) / (d['bb_upper'] - d['bb_lower'])

# -- Price Range Features --
d['hl_range'] = (d['high'] - d['low']) / d['close']
d['oc_range'] = (d['close'] - d['open']) / d['open']

# -- Volume Features --
if 'volume' in d.columns:
    d['volume_sma_10'] = d['volume'].rolling(10).mean()
    d['volume_ratio'] = d['volume'] / d['volume_sma_10']

# -- Lag Features (past values) --
for lag in [1, 2, 3, 5, 10]:
    d[f'close_lag_{lag}'] = d['close'].shift(lag)
    d[f'returns_lag_{lag}'] = d['returns'].shift(lag)

# -- Rolling Window Statistics --
for window in [5, 10, 20]:
    d[f'close_mean_{window}'] = d['close'].rolling(window).mean()
    d[f'close_std_{window}'] = d['close'].rolling(window).std()
    d[f'close_min_{window}'] = d['close'].rolling(window).min()
    d[f'close_max_{window}'] = d['close'].rolling(window).max()

# -- Time-Based Features --
d['day_of_week'] = d.index.dayofweek
d['month'] = d.index.month
d['quarter'] = d.index.quarter

# Target variable: next day's close price (for alt single-step training)
d['target'] = d['close'].shift(-1)

# Drop NaN rows from rolling calculations
d = d.dropna()

# Define the 31 features used by the model
FEATURE_COLUMNS = [
    'open', 'high', 'low', 'close', 'volume',
    'sma_10', 'sma_20', 'ema_10', 'ema_20',
    'returns', 'volatility_10', 'volatility_20',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'bb_position', 'hl_range', 'oc_range',
    'volume_ratio',
    'close_lag_1', 'close_lag_2', 'close_lag_3',
    'returns_lag_1', 'returns_lag_2',
    'close_mean_5', 'close_mean_10', 'close_std_10',
    'day_of_week', 'month',
]
# Keep only columns that actually exist
FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c in d.columns]

print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Total rows after feature engineering: {len(d)}")
print(f"Target: daily returns (stationary)")
print(f"Target range: [{d['returns'].min():.4f}, {d['returns'].max():.4f}]")

In [ ]:
# ===== CELL 5: Train-Test Split + Scaling =====
# Using 95/5 split (as per the research paper)
# Target = 'returns' (daily % change) — more stationary than raw prices

SEQ_LENGTH = 30       # Use 30 past days to predict the future
FORECAST_HORIZON = 30 # Predict 30 future returns at once (direct multi-step)

split_idx = int(len(d) * 0.95)
train_df = d.iloc[:split_idx]
test_df = d.iloc[split_idx:]
test_dates = test_df.index

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Scale features and target separately using MinMaxScaler
feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler = MinMaxScaler(feature_range=(0, 1))

train_features_scaled = feature_scaler.fit_transform(train_df[FEATURE_COLUMNS].values)
train_target_scaled = target_scaler.fit_transform(train_df[['returns']].values).ravel()

test_features_scaled = feature_scaler.transform(test_df[FEATURE_COLUMNS].values)
test_target_scaled = target_scaler.transform(test_df[['returns']].values).ravel()

print(f"Features scaled — Train: {train_features_scaled.shape}, Test: {test_features_scaled.shape}")

In [ ]:
# ===== CELL 6: Sequence Creation Helper =====
# Creates sliding windows for LSTM input
# X = sequence of 30 days of features
# y = next 30 days of returns (direct multi-step forecast)

def create_sequences(data, target, seq_length, forecast_horizon):
    """
    Converts flat data into overlapping sequences.
    X shape: (num_sequences, seq_length, n_features)
    y shape: (num_sequences, forecast_horizon)
    """
    X, y = [], []
    for i in range(len(data) - seq_length - forecast_horizon + 1):
        X.append(data[i : i + seq_length])
        y.append(target[i + seq_length : i + seq_length + forecast_horizon])
    return np.array(X), np.array(y)

In [ ]:
# ===== CELL 7: LSTM-Attention Model (PyTorch) =====
# Architecture (from the research paper):
#   LSTM(50 hidden units) → Multi-Head Attention (4 heads) → Flatten → Dense(output)
# This was the best performer in the paper with R² = 0.92

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism."""
    def __init__(self, num_heads, key_dim, output_dim=None):
        super().__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.output_dim = output_dim or key_dim * num_heads

        self.W_q = nn.Linear(key_dim, self.output_dim)
        self.W_k = nn.Linear(key_dim, self.output_dim)
        self.W_v = nn.Linear(key_dim, self.output_dim)
        self.W_o = nn.Linear(self.output_dim, self.output_dim)
        self.scale = torch.sqrt(torch.FloatTensor([key_dim]))

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        batch_size, seq_len = Q.size(0), Q.size(1)
        head_dim = self.output_dim // self.num_heads

        Q = Q.view(batch_size, seq_len, self.num_heads, head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale.to(Q.device)
        attention = torch.softmax(scores, dim=-1)
        attended = torch.matmul(attention, V)

        attended = attended.transpose(1, 2).contiguous().view(batch_size, seq_len, self.output_dim)
        return self.W_o(attended)


class LSTMAttentionModel(nn.Module):
    """
    LSTM-Attention Combined Model.
    Architecture: LSTM → Multi-Head Self-Attention → LayerNorm → Dense
    Supports direct multi-step forecasting via output_size > 1.
    """
    def __init__(self, input_size, hidden_size=50, num_heads=4,
                 key_dim=50, dropout=0.2, output_size=1):
        super().__init__()

        # Ensure hidden_size is divisible by num_heads
        if hidden_size % num_heads != 0:
            hidden_size = (hidden_size // num_heads) * num_heads
            if hidden_size == 0:
                hidden_size = num_heads

        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.dropout2 = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # LSTM encodes the sequence
        lstm_out, _ = self.lstm(x)
        lstm_out = self.dropout1(lstm_out)

        # Multi-Head Self-Attention with residual connection + layer norm
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        attn_out = self.dropout2(attn_out)
        attn_out = self.layer_norm(attn_out + lstm_out)

        # Take last timestep output and project to predictions
        out = attn_out[:, -1, :]
        return self.fc(out)

In [ ]:
# ===== CELL 8: Walk-Forward Cross Validation (5 folds) =====
# Trains on expanding windows and picks the best fold by R² score

n_splits = 5
train_min_size = int(len(d) * 0.7)
test_size = int((len(d) - train_min_size) / n_splits)

print(f"Total data points: {len(d)}")
print(f"Initial train: {train_min_size}, Test per fold: {test_size}")
print(f"Number of folds: {n_splits}\n")

cv_results = []

for fold in range(n_splits):
    print(f"{'=' * 40}")
    print(f"FOLD {fold + 1}/{n_splits}")
    print(f"{'=' * 40}")

    # Expanding window: each fold uses more training data
    train_end = train_min_size + fold * test_size
    test_end = train_end + test_size

    cv_train_df = d.iloc[:train_end]
    cv_test_df = d.iloc[train_end:test_end]

    # Scale for this fold
    cv_feat_scaler = MinMaxScaler()
    cv_tgt_scaler = MinMaxScaler()

    cv_train_feat = cv_feat_scaler.fit_transform(cv_train_df[FEATURE_COLUMNS].values)
    cv_train_tgt = cv_tgt_scaler.fit_transform(cv_train_df[['returns']].values).ravel()
    cv_test_feat = cv_feat_scaler.transform(cv_test_df[FEATURE_COLUMNS].values)
    cv_test_tgt = cv_tgt_scaler.transform(cv_test_df[['returns']].values).ravel()

    # Create sequences
    cv_X_train, cv_y_train = create_sequences(cv_train_feat, cv_train_tgt, SEQ_LENGTH, FORECAST_HORIZON)

    cv_ctx = cv_train_feat[-SEQ_LENGTH:]
    cv_X_full = np.vstack([cv_ctx, cv_test_feat])
    cv_y_full = np.concatenate([cv_train_tgt[-SEQ_LENGTH:], cv_test_tgt])
    cv_X_test, cv_y_test = create_sequences(cv_X_full, cv_y_full, SEQ_LENGTH, FORECAST_HORIZON)

    print(f"  Train: {cv_X_train.shape}, Test: {cv_X_test.shape}")

    # Build fresh model for this fold
    cv_model = LSTMAttentionModel(
        input_size=cv_X_train.shape[2],
        hidden_size=50, num_heads=4, key_dim=50,
        dropout=0.2, output_size=FORECAST_HORIZON
    ).to(device)

    # Split train into train/validation (90/10)
    cv_val_split = int(len(cv_X_train) * 0.9)
    cv_X_tr = cv_X_train[:cv_val_split]
    cv_y_tr = cv_y_train[:cv_val_split]
    cv_X_val = cv_X_train[cv_val_split:]
    cv_y_val = cv_y_train[cv_val_split:]

    # Create data loaders
    tr_loader = DataLoader(TensorDataset(torch.FloatTensor(cv_X_tr), torch.FloatTensor(cv_y_tr)),
                           batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.FloatTensor(cv_X_val), torch.FloatTensor(cv_y_val)),
                            batch_size=32, shuffle=False)

    # Training setup
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(cv_model.parameters(), lr=0.003)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-6)

    EPOCHS = 50
    PATIENCE_ES = 15
    best_vl = float('inf')
    best_st = None
    patience_cnt = 0

    print(f"  Training for up to {EPOCHS} epochs...")

    for epoch in range(EPOCHS):
        cv_model.train()
        tl, tm, nt = 0.0, 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            p = cv_model(xb)
            loss = criterion(p, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(cv_model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item() * len(xb)
            tm += torch.mean(torch.abs(p - yb)).item() * len(xb)
            nt += len(xb)

        cv_model.eval()
        vl, vm, nv = 0.0, 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                p = cv_model(xb)
                l = criterion(p, yb)
                vl += l.item() * len(xb)
                vm += torch.mean(torch.abs(p - yb)).item() * len(xb)
                nv += len(xb)

        avg_tl = tl / nt
        avg_vl = vl / nv
        scheduler.step(avg_vl)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {avg_tl:.6f} - Val Loss: {avg_vl:.6f}")

        if avg_vl < best_vl:
            best_vl = avg_vl
            best_st = {k: v.clone() for k, v in cv_model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE_ES:
                print(f"  Early stopping triggered at epoch {epoch+1}")
                break

    # Restore best weights
    if best_st is not None:
        cv_model.load_state_dict(best_st)

    # Evaluate this fold
    cv_model.eval()
    with torch.no_grad():
        cv_pred_scaled = cv_model(torch.FloatTensor(cv_X_test).to(device)).cpu().numpy()

    # Inverse transform to get actual returns
    cv_pred_returns = cv_tgt_scaler.inverse_transform(cv_pred_scaled.reshape(-1, 1)).reshape(-1, FORECAST_HORIZON)
    cv_actual_returns = cv_tgt_scaler.inverse_transform(cv_y_test.reshape(-1, 1)).reshape(-1, FORECAST_HORIZON)

    # Convert returns to prices (using first-step predictions)
    cv_all_close = cv_train_df['close'].values.tolist() + cv_test_df['close'].values.tolist()
    cv_prev_close = np.array([cv_all_close[i + SEQ_LENGTH - 1] for i in range(len(cv_X_test))])

    cv_actual_prices = cv_prev_close * (1 + cv_actual_returns[:, 0])
    cv_pred_prices = cv_prev_close * (1 + cv_pred_returns[:, 0])

    # Calculate metrics for this fold
    mae = mean_absolute_error(cv_actual_prices, cv_pred_prices)
    rmse = np.sqrt(mean_squared_error(cv_actual_prices, cv_pred_prices))
    r2 = r2_score(cv_actual_prices, cv_pred_prices)

    cv_results.append({
        'fold': fold + 1,
        'model': cv_model,
        'feature_scaler': cv_feat_scaler,
        'target_scaler': cv_tgt_scaler,
        'test_df': cv_test_df,
        'metrics': {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    })

    print(f"  Fold {fold + 1} — R²: {r2:.4f}, MAE: {mae:.2f}, RMSE: {rmse:.2f}")

# Select the best fold (highest R²)
best_fold = max(cv_results, key=lambda x: x['metrics']['R²'])

print(f"\n{'=' * 60}")
print(f"CV RESULTS SUMMARY")
print(f"{'=' * 60}")
print(f"{'Fold':<6} {'R²':>10} {'MAE':>10} {'RMSE':>10}")
print(f"{'-' * 40}")
for r in cv_results:
    m = r['metrics']
    print(f"{r['fold']:<6} {m['R²']:>10.4f} {m['MAE']:>10.2f} {m['RMSE']:>10.2f}")
print(f"{'-' * 40}")
print(f"Best fold: {best_fold['fold']} (R²={best_fold['metrics']['R²']:.4f})")
print(f"{'=' * 60}")

In [ ]:
# ===== CELL 9: Extract Best Model + Bias Correction =====
# Take the best model from CV and compute bias correction

model = best_fold['model']
feature_scaler = best_fold['feature_scaler']
target_scaler = best_fold['target_scaler']
test_df = best_fold['test_df']
fold_metrics = best_fold['metrics']

# Build test sequences for the best fold
best_fold_idx = best_fold['fold'] - 1
best_train_end = train_min_size + best_fold_idx * test_size
best_cv_train_df = d.iloc[:best_train_end]
best_cv_test_df = best_fold['test_df']

best_train_feat = feature_scaler.transform(best_cv_train_df[FEATURE_COLUMNS].values)
best_test_feat = feature_scaler.transform(best_cv_test_df[FEATURE_COLUMNS].values)

best_ctx = best_train_feat[-SEQ_LENGTH:]
best_X_full = np.vstack([best_ctx, best_test_feat])
best_y_full = np.concatenate([best_cv_train_df['returns'].values[-SEQ_LENGTH:], best_cv_test_df['returns'].values])
best_X_test, best_y_test = create_sequences(best_X_full, best_y_full, SEQ_LENGTH, FORECAST_HORIZON)

# Make multi-step predictions
model.eval()
with torch.no_grad():
    pred_scaled = model(torch.FloatTensor(best_X_test).to(device)).cpu().numpy()

pred_returns = target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(-1, FORECAST_HORIZON)
actual_returns = target_scaler.inverse_transform(best_y_test.reshape(-1, 1)).reshape(-1, FORECAST_HORIZON)

# Convert first-step returns to prices
all_close = np.concatenate([best_cv_train_df['close'].values, best_cv_test_df['close'].values])
seq_end_idx = np.arange(SEQ_LENGTH, len(all_close))
prev_close = all_close[seq_end_idx - 1]

n_test = len(pred_returns)
raw_pred_prices = prev_close[-n_test:] * (1 + pred_returns[:, 0])
actual_prices = prev_close[-n_test:] * (1 + actual_returns[:, 0])

# Bias correction: scale predictions by mean percentage error
mean_error = np.mean((actual_prices - raw_pred_prices) / actual_prices)
pred_prices = raw_pred_prices / (1 - mean_error)

print(f"\nBias Correction:")
print(f"  Mean error (bias): {mean_error * 100:.2f}%")
print(f"  Correction factor: {1 / (1 - mean_error):.4f}x")

# Recalculate metrics with corrected predictions
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    true_dir = np.diff(y_true)
    pred_dir = np.diff(y_pred)
    dir_acc = np.mean((true_dir > 0) == (pred_dir > 0)) * 100 if len(y_true) > 1 else None
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R²': r2, 'Directional Accuracy (%)': dir_acc}

metrics = calculate_metrics(actual_prices, pred_prices)

print(f"\n{'=' * 50}")
print(f"CORRECTED METRICS (Best Fold #{best_fold['fold']})")
print(f"{'=' * 50}")
for k, v in metrics.items():
    if v is not None:
        if 'Accuracy' in k:
            print(f"{k:25s}: {v:.2f}%")
        elif 'MAPE' in k or 'R²' in k:
            print(f"{k:25s}: {v:.4f}")
        else:
            print(f"{k:25s}: {v:.4f}")

In [ ]:
# ===== CELL 10: Naive Baseline Comparison =====
# Compare against a "predict no change" baseline

y_naive = prev_close[-n_test:]
naive_metrics = calculate_metrics(actual_prices, y_naive)

print(f"{'=' * 50}")
print(f"BASELINE COMPARISON")
print(f"{'=' * 50}")
print(f"{'Metric':<20} {'LSTM-Attention':>15} {'Naive':>15}")
print(f"{'-' * 52}")
for k in ['MAE', 'RMSE', 'R²', 'MAPE']:
    lstm_v = metrics[k]
    naive_v = naive_metrics[k]
    print(f"{k:<20} {lstm_v:>15.4f} {naive_v:>15.4f}")
print(f"{'-' * 52}")
print(f"LSTM R²:  {metrics['R²']:.4f}")
print(f"Naive R²: {naive_metrics['R²']:.4f}")

In [ ]:
# ===== CELL 11: Visualizations =====
# Generate plots: predictions vs actual, training history, error distribution

os.makedirs('results', exist_ok=True)
dates_plot = best_fold['test_df'].index[-n_test:]

# 1. Predictions vs Actual
plt.figure(figsize=(14, 7))
plt.plot(dates_plot, actual_prices, label='Actual', lw=2, alpha=0.8)
plt.plot(dates_plot, pred_prices, label='Predicted', lw=2, alpha=0.8)
plt.ylabel('Price (USD)')
plt.xlabel('Date')
plt.title('Gold Price: LSTM-Attention Prediction vs Actual (1-step)', fontsize=16, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('results/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Error Distribution
errors = actual_prices - pred_prices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='r', ls='--', lw=2)
axes[0].set_title('Error Distribution')
axes[0].set_xlabel('Prediction Error')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

mn = min(actual_prices.min(), pred_prices.min())
mx = max(actual_prices.max(), pred_prices.max())
axes[1].scatter(actual_prices, pred_prices, alpha=0.5, edgecolors='none')
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title('Actual vs Predicted')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/error_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Save predictions CSV
predictions_df = pd.DataFrame({
    'Actual': actual_prices,
    'Predicted': pred_prices,
    'Error': actual_prices - pred_prices,
    'Error_%': ((actual_prices - pred_prices) / actual_prices) * 100,
})
predictions_df.index = dates_plot
predictions_df.to_csv('results/predictions.csv')

print("Results saved:")
print(f"  - results/predictions_vs_actual.png")
print(f"  - results/error_distribution.png")
print(f"  - results/predictions.csv")

In [ ]:
# ===== CELL 12: Final Summary =====

print("\n" + "=" * 60)
print("PIPELINE COMPLETE — SUMMARY")
print("=" * 60)
print(f"\nModel: LSTM-Attention — Direct Multi-Step (H={FORECAST_HORIZON})")
print(f"Validation: Walk-Forward Cross Validation (5 folds)")
print(f"Best Fold: #{best_fold['fold']}")
print(f"\nFinal Metrics (1-step-ahead, bias-corrected):")
for k, v in metrics.items():
    if v is not None:
        if 'Accuracy' in k:
            print(f"  {k}: {v:.2f}%")
        else:
            print(f"  {k}: {v:.4f}")

print(f"\n{'=' * 60}")
print(f"DONE")
print(f"{'=' * 60}")